# Task 2: Text Chunking, Embedding, and Vector Store Indexing

This notebook demonstrates the Task 2 pipeline for CrediTrust's complaint RAG system:

1. **Stratified sampling** (~12K complaints) preserving product-category proportions
2. **Text chunking** with 500-character windows and 50-character overlap
3. **Embedding** with `sentence-transformers/all-MiniLM-L6-v2`
4. **Indexing** into a persistent ChromaDB store under `vector_store/chromadb/`

> **Prerequisite:** Run Task 1 first so `data/processed/complaints_clean.csv` exists.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import pandas as pd

from src.chunking import chunk_text
from src.config import (
    DEFAULT_CHUNK_OVERLAP,
    DEFAULT_CHUNK_SIZE,
    DEFAULT_SAMPLE_SIZE,
    TASK2_DATA_PATH,
    TASK2_SAMPLE_PATH,
    TASK2_VECTOR_STORE_PATH,
)
from src.sampling import sampling_summary, stratified_sample

pd.set_option("display.max_colwidth", 120)

## 1. Load cleaned complaints

We use the filtered, cleaned dataset produced in Task 1.

In [ ]:
df = pd.read_csv(TASK2_DATA_PATH)
print(f"Total cleaned complaints: {len(df):,}")
print(df["product_category"].value_counts())
df[["product_category", "clean_narrative"]].head(3)

## 2. Stratified sampling strategy

We draw **12,000 complaints** with `sklearn.model_selection.train_test_split(..., stratify=product_category)` so each of the four target products keeps roughly the same share as in the full cleaned dataset.

**Why stratified?** Credit Card complaints dominate the corpus; random sampling would under-represent Personal Loan and Money Transfer narratives, hurting retrieval quality for those products.

In [ ]:
sample_df = stratified_sample(df, n=DEFAULT_SAMPLE_SIZE, random_state=42)
sample_df.to_csv(TASK2_SAMPLE_PATH, index=False)

print(f"Sample size: {len(sample_df):,}")
print("\nOriginal proportions:")
display(sampling_summary(df))
print("\nSample proportions:")
display(sampling_summary(sample_df))

## 3. Chunking experiments

Long complaint narratives are split before embedding. We mirror the pre-built full-scale index:

| Parameter | Value | Rationale |
| --- | --- | --- |
| `chunk_size` | 500 chars | Fits typical complaint excerpts; aligns with pre-built store |
| `chunk_overlap` | 50 chars | Preserves context across chunk boundaries |

The splitter follows LangChain's `RecursiveCharacterTextSplitter` priority: paragraphs → lines → sentences → words.

In [ ]:
sample_narrative = sample_df.iloc[0]["clean_narrative"]

configs = [
    (400, 40),
    (DEFAULT_CHUNK_SIZE, DEFAULT_CHUNK_OVERLAP),
    (700, 70),
]

for size, overlap in configs:
    chunks = chunk_text(sample_narrative, chunk_size=size, chunk_overlap=overlap)
    avg_len = sum(len(c.text) for c in chunks) / max(len(chunks), 1)
    print(f"size={size}, overlap={overlap} -> {len(chunks)} chunks, avg length {avg_len:.0f}")

chosen = chunk_text(sample_narrative, chunk_size=DEFAULT_CHUNK_SIZE, chunk_overlap=DEFAULT_CHUNK_OVERLAP)
print("\nFirst chunk (500/50):")
print(chosen[0].text[:300], "...")

## 4. Build embeddings and ChromaDB index

**Embedding model:** `sentence-transformers/all-MiniLM-L6-v2`
- 384-dimensional dense vectors
- Fast on CPU, strong semantic search for short-to-medium financial text
- Same model used in the pre-built full-scale vector store for Tasks 3–4

Run the CLI script below (recommended) or call `build_vector_store.main()` from a code cell.

In [ ]:
# Option A: run from terminal
# !python -m src.build_vector_store --sample-size 12000

# Option B: dry-run (sample + chunk only, no embedding)
# !python -m src.build_vector_store --dry-run

print("Expected outputs:")
print(f"  Sample CSV: {TASK2_SAMPLE_PATH}")
print(f"  Vector store: {TASK2_VECTOR_STORE_PATH}")
print(f"  Build stats: {TASK2_VECTOR_STORE_PATH / 'build_stats.json'}")

## 5. Task 2 summary (for report)

**Sampling:** Stratified 12K sample via sklearn, preserving Credit Card / Personal Loan / Savings Account / Money Transfer proportions.

**Chunking:** 500 characters with 50-character overlap using recursive character splitting; balances retrieval precision with enough local context for billing, fraud, and service issues.

**Embeddings:** `all-MiniLM-L6-v2` for speed and compatibility with the full pre-built index used in Tasks 3–4.

**Storage:** ChromaDB persistent collection `complaint_chunks` with per-chunk metadata (`complaint_id`, `product_category`, `issue`, `chunk_index`, `total_chunks`).